# About
- A `TinyLM`, autoregressive model for a simple storyteller.

# Setup

In [1]:
from pprint import pprint
import os
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import set_seed
from transformers.utils import logging

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Seed
seed = 42
set_seed(seed)

# Suppress loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1012 15:28:50.813000 5440 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1012 15:28:50.879000 5440 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1012 15:28:51.790000 5440 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton k

# 1. Dataset

- A short story dataset generated by `gpt-3.5` and `gpt-4`.
- Size: `n_train=2,119,719` and `n_valid=21,990`
- Intentionally used a simple and small vocabulary.
- Ideal dataset to test if a small LM could learn grammar, basic reasoning, storytelling, etc.

In [2]:
from datasets import load_dataset, DatasetDict

data = load_dataset('roneneldan/TinyStories')

ds = DatasetDict({
    'train': data['train'],
    'valid': data['validation'],
})

pprint(ds['train'][0])

{'text': 'One day, a little girl named Lily found a needle in her room. She '
         'knew it was difficult to play with it because it was sharp. Lily '
         'wanted to share the needle with her mom, so she could sew a button '
         'on her shirt.\n'
         '\n'
         'Lily went to her mom and said, "Mom, I found this needle. Can you '
         'share it with me and sew my shirt?" Her mom smiled and said, "Yes, '
         'Lily, we can share the needle and fix your shirt."\n'
         '\n'
         "Together, they shared the needle and sewed the button on Lily's "
         'shirt. It was not difficult for them because they were sharing and '
         'helping each other. After they finished, Lily thanked her mom for '
         'sharing the needle and fixing her shirt. They both felt happy '
         'because they had shared and worked together.'}


# 2. Tokenizer

- Normalizer: `Lowercase` + `Strip`
- Pre-tokenizer: `Whitespace`
- Model: Character-level BPE
- Post-processor
  - `<bos>` / `<eos>`
  - `<unk>`

In [3]:
from tokenizers import (
    Tokenizer,
    pre_tokenizers,
    models,
    processors,
    trainers,
)
from tokenizers.normalizers import Sequence, Lowercase, Strip
from transformers import PreTrainedTokenizerFast


# Steps
normalizer = Sequence([Lowercase(), Strip()])       # Normalizer
pre_tokenizer = pre_tokenizers.Whitespace()         # Pre-tokenizer
tokenizer_model = models.BPE(unk_token='<unk>')               # Model
processor = processors.TemplateProcessing(          # Processor
    single='<bos> $A <eos>',
    special_tokens=[('<bos>', 0), ('<eos>', 1)],
)

# Init
tokenizer = Tokenizer(model=tokenizer_model)
tokenizer.normalizer = normalizer
tokenizer.pre_tokenizer = pre_tokenizer
tokenizer.post_processor = processor

special_tokens = ['<bos>', '<eos>', '<unk>']

# Train
n_train_tokenize = 100_000
ds_train_tokenizer = ds['train'][:n_train_tokenize]['text']
max_token_length = 10

trainer = trainers.BpeTrainer(
    vocab_size=10_000,      # maximum vocab size.
    min_frequency=2,
    special_tokens=special_tokens,

    limit_alphabet=50,                      # retain the 50 most frequent characters (except special tokens) and continue merging.
    max_token_length=max_token_length,      # maximum length of learned token.

    # initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),       # starts with 256 unicode symbols.

    continuing_subword_prefix='##',     # prepend to subwords that do not start a word.
    end_of_word_suffix='',              # append to subwords that end a word.

    show_progress=True,
)

tokenizer.train_from_iterator(
    tqdm(ds_train_tokenizer),
    trainer,
    length=n_train_tokenize,
)

vocab = tokenizer.get_vocab()
V = len(vocab)

# Wrapper
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token='<unk>',
    bos_token='<bos>',
    eos_token='<eos>',
    pad_token='<pad>',
    model_max_length=max_token_length,
)

100%|██████████| 100000/100000 [00:01<00:00, 75473.91it/s]


In [4]:
n_train = 100_000       # train size for LLM.
train_ds = ds['train'].select(range(n_train))

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=256,
    )

tokenized = train_ds.map(tokenize, batched=True, remove_columns=['text'])

Map: 100%|██████████| 100000/100000 [00:07<00:00, 14042.49 examples/s]


# 3. Transformer

## 3.1. Embedding

- Word embedding: (V, d)
- Positional embedding: RoPE

In [5]:
from transformers import AutoModel

d = 512
model_qwen = AutoModel.from_pretrained('Qwen/Qwen2.5-0.5B')

word_emb = nn.Embedding(num_embeddings=V, embedding_dim=d)
pos_emb = model_qwen.rotary_emb

## 3.2. Attention

- Multi-head attention
  - $H$: `n_heads`
  - $d\_head = d / H$

In [6]:
from transformers import AutoConfig
from transformers.models.qwen2.modeling_qwen2 import (
    Qwen2RotaryEmbedding,
    apply_rotary_pos_emb,
)

class TinyAttention(nn.Module):
    def __init__(self, d, H, p):
        super().__init__()

        self.d = d      # embedding dimension.
        self.H = H      # n_heads for multi-head attention.
        self.p = p      # attention dropout ratio.
        self.d_head = d // H

        self.wq = nn.Linear(d, d)
        self.wk = nn.Linear(d, d)
        self.wv = nn.Linear(d, d)
        self.attn_dropout = nn.Dropout(p=p)
        self.attn_out = nn.Linear(d, d)
        self.attn_norm = nn.LayerNorm(d)

        config = AutoConfig.from_pretrained('Qwen/Qwen2.5-0.5B')
        config.hidden_size = d
        config.num_attention_heads = H
        config.num_key_value_heads = H
        config.head_dim = self.d_head
        self.rope_emb = Qwen2RotaryEmbedding(config=config)

    def forward(self, x, attention_mask=None):
        B, L = x.shape[:2]  # B = n_seqs, L = n_tokens.
        d = self.d
        H = self.H

        # QKV projection
        q = self.wq(x)      # (B, L, d)
        k = self.wk(x)
        v = self.wv(x)

        # Multi-head attention
        q = q.reshape(B, L, H, self.d_head).transpose(1, 2)      # (B, L, d) -> (B, L, H, d_head) -> (B, H, L, d_head)
        k = k.reshape(B, L, H, self.d_head).transpose(1, 2)          
        v = v.reshape(B, L, H, self.d_head).transpose(1, 2)  

        # RoPE
        position_ids = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)

        cos, sin = self.rope_emb(q, position_ids)       # (B, L, d_head)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)     # (B, H, L, d_head)        

        # Attention scores
        scores = q @ k.transpose(-2, -1)        # k: (B, H, d_head, L), scores: (B, H, L, L)
        scores = scores / (self.d_head ** 0.5)

        # Causal mask
        mask = torch.triu(
            torch.ones(L, L, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        scores = scores.masked_fill(mask, float("-inf"))

        # Padding mask
        if attention_mask is not None:
            padding_mask = attention_mask[:, None, None, :] == 0
            scores = scores.masked_fill(padding_mask, torch.finfo(scores.dtype).min)

        # Attention weights
        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)    # softmax(QK^T) ---dropout--- softmax(QK^T) * V

        # Attention out
        out = weights @ v               # (B, H, L, d_head)
        out = out.transpose(1, 2)       # (B, L, H, d_head)
        out = out.reshape(B, L, d)      # (B, L, d)
        out = self.attn_out(out)        # (B, L, d)

        return out

## 3.3. Feedforward Network

- $(d, d_{ffn}) \rightarrow (d_{ffn}, d)$

> Note) For standard transformers, $d_{ffn} \approx 4 \times d$.

In [7]:
class TinyFFN(nn.Module):
    def __init__(self, d, d_ffn):
        super().__init__()

        self.d = d
        self.d_ffn = d_ffn

        self.hidden_1   = nn.Linear(d, d_ffn)
        self.hidden_2   = nn.Linear(d_ffn, d)
        self.activation = nn.GELU()

    def forward(self, x):
        x = self.hidden_1(x)
        x = self.activation(x)
        x = self.hidden_2(x)

        return x

## 3.4. Transformer Block

```markdown
x
│
├───────────────────┐
↓                   │
LayerNorm           │
↓                   │
Attention           │ residual
↓                   │
+ ←─────────────────┘
│
├───────────────────┐
↓                   │
LayerNorm           │
↓                   │
FFN                 │ residual
↓                   │
+ ←─────────────────┘
│
↓
output
```

In [8]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, d, H, p):
        super().__init__()

        self.d = d
        self.H = H
        self.p = p

        self.norm_1 = nn.LayerNorm(d)
        self.attn = TinyAttention(d, H, p)

        self.norm_2 = nn.LayerNorm(d)
        self.ffn = TinyFFN(d, d*4)

    def forward(self, x, attention_mask=None):
        # Attention + residual
        x = x + self.attn(self.norm_1(x), attention_mask=attention_mask)

        # FFN + residual
        x = x + self.ffn(self.norm_2(x))

        return x

## 3.5. Transformer

```markdown
Token Embedding + Position Embedding
              │
              ↓
┌──────────────────────────────┐
│ Transformer Block 1          │
│                              │
│  LayerNorm → Attention → +   │
│  LayerNorm → FFN       → +   │
└──────────────────────────────┘
              ↓
┌──────────────────────────────┐
│ Transformer Block 2          │
│                              │
│  LayerNorm → Attention → +   │
│  LayerNorm → FFN       → +   │
└──────────────────────────────┘
              ↓
             ...
              ↓
┌──────────────────────────────┐
│ Transformer Block N          │
│                              │
│  LayerNorm → Attention → +   │
│  LayerNorm → FFN       → +   │
└──────────────────────────────┘
              ↓
        Final LayerNorm
              ↓
          LM Head
              ↓
           Logits
```

In [9]:
class TinyLM(nn.Module):
    def __init__(self, V, d, H, p, N=1):
        super().__init__()

        self.V = V
        self.d = d
        self.H = H
        self.p = p
        self.N = N

        # Embedding
        self.word_emb = nn.Embedding(num_embeddings=V, embedding_dim=d)

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TinyTransformerBlock(d, H, p) for _ in range(N)
        ])

        # Output
        self.final_norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, V)

    def forward(self, input_ids, attention_mask=None, labels=None):
        # Embedding
        x = self.word_emb(input_ids)

        # Transformer blocks
        for block in self.blocks:
            x = block(x, attention_mask=attention_mask)

        # Output
        x = self.final_norm(x)
        logits = self.lm_head(x)

        # Loss
        if labels is not None:
            shift_logits = logits[:, :-1, :]
            shift_labels = labels[:, 1:]

            loss = F.cross_entropy(
                shift_logits.reshape(-1, self.V),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

            return {"loss": loss, "logits": logits}

        return {"logits": logits}

## 3.6. HF Wrapper

In [10]:
from transformers import (
    PreTrainedModel,
    PreTrainedConfig,
    GenerationMixin,
    GenerationConfig,
)
from transformers.modeling_outputs import CausalLMOutput

class TinyLMConfig(PreTrainedConfig):
    model_type = "tinylm"

    def __init__(self, vocab_size=10000, d=128, H=4, p=0.1, N=1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)

        self.d = d
        self.H = H
        self.p = p
        self.N = N

class TinyLMHF(PreTrainedModel, GenerationMixin):
    config_class = TinyLMConfig

    def __init__(self, config, model=None):
        super().__init__(config)

        self.model = model if model is not None else TinyLM(
            V=config.vocab_size,
            d=config.d,
            H=config.H,
            p=config.p,
            N=config.N,
        )

        self.generation_config = GenerationConfig.from_model_config(config)
        self.generation_config.use_cache = False

        # Preserve the weights of an existing TinyLM when wrapping it.
        if model is not None:
            for module in self.model.modules():
                module._is_hf_initialized = True

        self.post_init()    # Required Hugging Face initialization.

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        return CausalLMOutput(
            loss=outputs.get("loss"),
            logits=outputs["logits"],
        )

## 3.7. Model

In [11]:
model_raw = TinyLM(
    V=len(tokenizer),
    d=128,
    H=4,
    p=0.1,
    N=1,
).to(device)

config = TinyLMConfig(
    vocab_size=model_raw.V,
    d=model_raw.d,
    H=model_raw.H,
    p=model_raw.p,
    N=model_raw.N,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = TinyLMHF(config, model=model_raw)

print(model)

TinyLMHF(
  (model): TinyLM(
    (word_emb): Embedding(10001, 128)
    (blocks): ModuleList(
      (0): TinyTransformerBlock(
        (norm_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): TinyAttention(
          (wq): Linear(in_features=128, out_features=128, bias=True)
          (wk): Linear(in_features=128, out_features=128, bias=True)
          (wv): Linear(in_features=128, out_features=128, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (attn_out): Linear(in_features=128, out_features=128, bias=True)
          (attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (rope_emb): Qwen2RotaryEmbedding()
        )
        (norm_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (ffn): TinyFFN(
          (hidden_1): Linear(in_features=128, out_features=512, bias=True)
          (hidden_2): Linear(in_features=512, out_features=128, bias=True)
          (activation)

# 4. Training

In [12]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # clm -> creates `labels` from `input_ids`, and paddings -> -100.
)

In [18]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../tmp/tinylm",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    learning_rate=3e-4,
    weight_decay=1e-2,
    logging_steps=500,
    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=collator,
    processing_class=tokenizer,
)

training_results = trainer.train()

pprint(training_results.metrics)

Step,Training Loss
500,3.247611
1000,3.191708
1500,3.136112
2000,3.093235
2500,3.067910
3000,3.049784
3500,3.024043
4000,3.015006
4500,3.006760


{'epoch': 3.0,
 'total_flos': 686091110400000.0,
 'train_loss': 3.088713421600921,
 'train_runtime': 168.977,
 'train_samples_per_second': 1775.389,
 'train_steps_per_second': 27.749}


# 5. Generation

In [19]:
model.eval()

inputs = tokenizer(
    "<bos>Once upon a time",
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        use_cache=False,
    )

pprint(tokenizer.decode(outputs[0], skip_special_tokens=True))

('once upon a time there was a little girl named lily . she loved to explore , '
 'but her mom made her happy and had big eyes . one day , lily was playing '
 "with her crayons . lily loved playing with her dolls , but she didn ' t want "
 'to come inside . she tried to jump in the kitchen and touch . later , she '
 'kept a dream . her mom would remember how to play with her on her head and '
 'they were all very hard and had a great time . lily was happy and thanked '
 'her dad . she hugged her mom . lily and said , " i like it \' s okay to play '
 'with you ." lily was scared , but she still couldn \' t get to be mad at '
 "drawing . but then , lily ' s dad started to cry and she started to cry . "
 'lily was sad , but she knew she had to wear her toys and never forgot how '
 "the doll ' s favorite color .")


# 6. Save / Load

In [20]:
# Save
path = "../tmp/tinylm"

model.save_pretrained(path)
tokenizer.save_pretrained(path)

# Load
model = TinyLMHF.from_pretrained(path).to(device)
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

model.eval()

TinyLMHF(
  (model): TinyLM(
    (word_emb): Embedding(10001, 128)
    (blocks): ModuleList(
      (0): TinyTransformerBlock(
        (norm_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): TinyAttention(
          (wq): Linear(in_features=128, out_features=128, bias=True)
          (wk): Linear(in_features=128, out_features=128, bias=True)
          (wv): Linear(in_features=128, out_features=128, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (attn_out): Linear(in_features=128, out_features=128, bias=True)
          (attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (rope_emb): Qwen2RotaryEmbedding()
        )
        (norm_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (ffn): TinyFFN(
          (hidden_1): Linear(in_features=128, out_features=512, bias=True)
          (hidden_2): Linear(in_features=512, out_features=128, bias=True)
          (activation)